In [5]:
pip install pandas

   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   ------------------------------- -------- 7.9/9.9 MB 61.4 MB/s eta 0:00:01
   ---------------------------------------- 9.9/9.9 MB 47.4 MB/s  0:00:00
   ---------------------------------------- 0.0/12.4 MB ? eta -:--:--
   ---------------------------------------  12.3/12.4 MB 70.8 MB/s eta 0:00:01
   ---------------------------------------- 12.4/12.4 MB 54.0 MB/s  0:00:00

   ---------------------------------------- 0/3 [tzdata]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [numpy]
   ------------- -------------------------- 1/3 [

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
pip install openpyxl


   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   -------------------- ------------------- 1/2 [openpyxl]
   ---------------------------------------- 2/2 [openpyxl]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import pandas as pd

neighborhoods = pd.read_csv(r"C:\Users\ianre\OneDrive\Documents\UB\Extracurricular\SPARQL\CSVtoRDF\Minneapolis_Neighborhoods_-2997528892731440104.csv")
crime = pd.read_excel(r"C:\Users\ianre\OneDrive\Documents\UB\Extracurricular\SPARQL\CSVtoRDF\Crime_Data.xlsx", sheet_name="Subset")

In [13]:
print(neighborhoods.columns.tolist())
print(crime.columns.tolist())

['OBJECTID', 'INT_REFNO', 'PREFIX', 'UDI', 'SYMBOL_NAM', 'BDNAME', 'BDNUM', 'TEXT_NBR', 'Shape__Area', 'Shape__Length']
['X', 'Y', 'Type', 'Case_Number', 'Case_NumberAlt', 'Reported_Date', 'Occurred_Date', 'NIBRS_Crime_Against', 'NIBRS_Group', 'NIBRS_Code', 'Offense_Category', 'Offense', 'Problem_Initial', 'Problem_Final', 'Address', 'Precinct', 'Neighborhood', 'Ward', 'Latitude', 'Longitude', 'wgsXAnon', 'wgsYAnon', 'Crime_Count', 'OBJECTID']


In [15]:
pip install rdflib

   ---------------------------------------- 0.0/615.4 kB ? eta -:--:--
   ---------------------------------------- 615.4/615.4 kB 10.3 MB/s  0:00:00

   ---------------------------------------- 0/2 [pyparsing]
   -------------------- ------------------- 1/2 [rdflib]
   -------------------- ------------------- 1/2 [rdflib]
   -------------------- ------------------- 1/2 [rdflib]
   -------------------- ------------------- 1/2 [rdflib]
   -------------------- ------------------- 1/2 [rdflib]
   -------------------- ------------------- 1/2 [rdflib]
   -------------------- ------------------- 1/2 [rdflib]
   ---------------------------------------- 2/2 [rdflib]

Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [28]:
import pandas as pd
from rdflib import Graph, Namespace, URIRef, Literal, RDF, OWL
from rdflib.namespace import XSD, RDFS

# Namespaces
MPLS = Namespace("https://ian-reinl.github.io/mpls-crime-ontology/ontology.ttl#")
BFO = Namespace("http://purl.obolibrary.org/obo/")
DATA = Namespace("https://ian-reinl.github.io/mpls-crime-ontology/data/")

# Build graph
g = Graph()
g.bind("mpls", MPLS)
g.bind("data", DATA)
g.bind("xsd", XSD)
g.bind("bfo", "http://purl.obolibrary.org/obo/")

# Load ontology
g.parse(r"C:\Users\ianre\OneDrive\Documents\UB\Extracurricular\SPARQL\CSVtoRDF\Crime&Neighborhoods.ttl", format="turtle")

print(f"Ontology loaded: {len(g)} triples")

Ontology loaded: 46 triples


In [29]:
# Load neighborhoods CSV
neighborhoods = pd.read_csv(r"C:\Users\ianre\OneDrive\Documents\UB\Extracurricular\SPARQL\CSVtoRDF\Minneapolis_Neighborhoods_-2997528892731440104.csv")

# Map each neighborhood row to RDF
for _, row in neighborhoods.iterrows():
    # Mint a URI for each neighborhood individual using BDNUM as the identifier
    neighborhood_uri = DATA[f"neighborhood/{row['BDNUM']}"]
    
    g.add((neighborhood_uri, RDF.type, MPLS.MinneapolisNeighborhood))
    g.add((neighborhood_uri, MPLS.neighborhoodName, Literal(row["BDNAME"], datatype=XSD.string)))
    g.add((neighborhood_uri, MPLS.neighborhoodNumber, Literal(int(row["BDNUM"]), datatype=XSD.integer)))
    g.add((neighborhood_uri, MPLS.areaSize, Literal(float(row["Shape__Area"]), datatype=XSD.decimal)))

print(f"Neighborhoods mapped: {len(neighborhoods)} rows")
print(f"Total triples so far: {len(g)}")

Neighborhoods mapped: 87 rows
Total triples so far: 394


In [30]:
# Load crime data
crime = pd.read_excel(r"C:\Users\ianre\OneDrive\Documents\UB\Extracurricular\SPARQL\CSVtoRDF\Crime_Data.xlsx", sheet_name="Subset")

# Map each crime row to RDF
for _, row in crime.iterrows():
    # Mint a URI for each incident using Case_Number as identifier
    incident_uri = DATA[f"incident/{row['Case_Number']}"]
    
    # Find the matching neighborhood URI by name
    neighborhood_match = neighborhoods[neighborhoods["BDNAME"] == row["Neighborhood"]]
    
    g.add((incident_uri, RDF.type, MPLS.CrimeIncident))
    g.add((incident_uri, MPLS.caseNumber, Literal(str(row["Case_Number"]), datatype=XSD.string)))
    g.add((incident_uri, MPLS.reportedDate, Literal(str(row["Reported_Date"]).replace("/", "-"), datatype=XSD.dateTime)))
    g.add((incident_uri, MPLS.occurredDate, Literal(str(row["Occurred_Date"]).replace("/", "-"), datatype=XSD.dateTime)))
    g.add((incident_uri, MPLS.offense, Literal(str(row["Offense"]), datatype=XSD.string)))
    g.add((incident_uri, MPLS.address, Literal(str(row["Address"]), datatype=XSD.string)))
    g.add((incident_uri, MPLS.precinct, Literal(int(row["Precinct"]), datatype=XSD.integer)))
    g.add((incident_uri, MPLS.ward, Literal(int(row["Ward"]), datatype=XSD.integer)))
    
    # Add occurs_in link if neighborhood match found
    if not neighborhood_match.empty:
        bdnum = neighborhood_match.iloc[0]["BDNUM"]
        neighborhood_uri = DATA[f"neighborhood/{bdnum}"]
        g.add((incident_uri, BFO["BFO_0000066"], neighborhood_uri))

print(f"Incidents mapped: {len(crime)} rows")
print(f"Total triples: {len(g)}")

Incidents mapped: 102 rows
Total triples: 1312


In [31]:
g.serialize("output.ttl", format="turtle")
print("Done")

Done
